# 第14回：Kaggle改善会

**今日の問い：限られた時間で、次に何を試すか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 限られた時間で実験を優先順位付けし、OOFスタッキングで統合する
- adversarial validationで学習とテストの分布ずれを点検する
- 複数シードの平均と閾値調整で、偶然に頼らない改善を積む

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- OOFスタッキング：OOF予測を入力に上位モデルで統合する方法
- 分布ずれ：学習とテストで入力の分布が違うこと
- シードアンサンブル：乱数だけ変えた複数モデルの平均
- 閾値調整：確率からクラスへの境界を変えること
- 実験統合：有効な変更を再検証しながら組み合わせること

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd
train = pd.read_csv(DATA / "local_competition" / "train.csv")
test = pd.read_csv(DATA / "local_competition" / "test.csv")
answers = pd.read_csv(DATA / "local_competition" / "instructor_answers.csv")


## 5人の担当

1. 欠損補完 / 2. 特徴量（最適温度からの距離） / 3. モデルの深さ / 4. 判定閾値 / 5. 誤分類の確認

全員が同じ`random_state=42`とF1を使い、担当箇所以外は変えません。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

improved_train = train.copy()
improved_test = test.copy()
for frame in [improved_train, improved_test]:
    frame["temperature_distance"] = (frame["temperature_c"] - 78).abs()
target = "active"
ignored = ["sample_id", "experiment_date", "smiles", target]
features = [c for c in improved_train.columns if c not in ignored]
numeric = improved_train[features].select_dtypes(include="number").columns.tolist()
categorical = [c for c in features if c not in numeric]
preprocess = ColumnTransformer([
    ("数値", SimpleImputer(strategy="median"), numeric),
    ("カテゴリ", Pipeline([("補完", SimpleImputer(strategy="most_frequent")), ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical),
])
model = Pipeline([("前処理", preprocess), ("モデル", RandomForestClassifier(n_estimators=300, max_depth=3, class_weight="balanced", random_state=42))])
X_train, X_valid, y_train, y_valid = train_test_split(improved_train[features], improved_train[target], test_size=0.25, random_state=42, stratify=improved_train[target])
model.fit(X_train, y_train)
print("改善案のローカルF1:", round(f1_score(y_valid, model.predict(X_valid)), 3))


In [ ]:
model.fit(improved_train[features], improved_train[target])
improved_submission = pd.DataFrame({"sample_id": improved_test["sample_id"], "active": model.predict(improved_test[features])})
merged = answers.merge(improved_submission, on="sample_id", suffixes=("_true", "_pred"))
print("模擬Leaderboard F1:", round(f1_score(merged["active_true"], merged["active_pred"]), 3))


## 実験ログ

改善しても悪化しても、`変更点 / ローカルF1 / 模擬Leaderboard F1 / 気づき`を1行で記録します。Leaderboardだけ改善し、ローカル検証が悪化した案は慎重に扱います。


## DEEP DIVE：OOFスタッキング・分布ずれ・シード平均

単体を超えるには、間違え方の違うモデルをOOFで束ね、分布ずれと偶然を点検します。


In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

pre = ColumnTransformer([
    ("n", SimpleImputer(strategy="median"), numeric),
    ("c", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore", sparse_output=False)), categorical),
])
members = {
    "rf": make_pipeline(pre, RandomForestClassifier(n_estimators=300, max_depth=4, random_state=42)),
    "hgb": make_pipeline(pre, HistGradientBoostingClassifier(max_iter=200, random_state=42)),
    "logit": make_pipeline(pre, LogisticRegression(max_iter=1000)),
}
skf = StratifiedKFold(5, shuffle=True, random_state=42)
oof = {}
for name, est in members.items():
    oof[name] = cross_val_predict(est, improved_train[features], improved_train[target], cv=skf, method="predict_proba")[:, 1]
    print(f"{name:6s} OOF F1:", round(f1_score(improved_train[target], (oof[name] >= 0.5).astype(int)), 3))
meta_X = pd.DataFrame(oof)
stack_oof = cross_val_predict(LogisticRegression(max_iter=1000), meta_X, improved_train[target], cv=skf, method="predict_proba")[:, 1]
print("スタッキング OOF F1:", round(f1_score(improved_train[target], (stack_oof >= 0.5).astype(int)), 3))


### 分布ずれを点検する


In [ ]:
from sklearn.model_selection import cross_val_score

combined = pd.concat([
    improved_train[numeric].assign(is_test=0),
    improved_test[numeric].assign(is_test=1),
], ignore_index=True)
adv = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, random_state=42))
auc = cross_val_score(adv, combined[numeric], combined["is_test"], cv=5, scoring="roc_auc")
print("adversarial validation AUC:", round(auc.mean(), 3), "（0.5付近なら分布は近い）")


### シード平均で偶然を薄める


In [ ]:
import numpy as np

probs = []
for seed in [0, 1, 2, 3, 4]:
    est = make_pipeline(pre, RandomForestClassifier(n_estimators=300, max_depth=4, random_state=seed)).fit(improved_train[features], improved_train[target])
    probs.append(est.predict_proba(improved_test[features])[:, 1])
ensemble_pred = (np.mean(probs, axis=0) >= 0.5).astype(int)
seed_merged = answers.merge(pd.DataFrame({"sample_id": improved_test["sample_id"], "active": ensemble_pred}), on="sample_id", suffixes=("_true", "_pred"))
print("5シード平均の模擬LB F1:", round(f1_score(seed_merged["active_true"], seed_merged["active_pred"]), 3))


## よくある誤り

- 5人の変更を一度に統合する
- Leaderboardだけを目的関数にする
- 分布ずれを無視してランダム分割だけで判断する

## SELF-STUDY（任意・30〜60分）

- 単体最良・投票・スタッキングのOOF F1を比較する
- adversarial validationのAUCが高い列を除いて再評価する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. OOFスタッキングの手順は何か
2. 分布ずれをどう検知するか
3. 改善を統合する順序はどうするか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
